In [0]:
# ════════════════════════════════════════════════════════════════
# run_framework_enhanced.py - COMPREHENSIVE METADATA SUPPORT
# ════════════════════════════════════════════════════════════════
# Supports ALL columns:
#   - DATA_FLOW_GROUP_ID, LOB, SOURCE, TARGET_OBJ_SCHEMA, TARGET_OBJ_NAME
#   - PRIORITY, TARGET_OBJ_TYPE, TRANSFORM_QUERY, GENERIC_SCRIPTS
#   - SOURCE_PK, TARGET_PK, LOAD_TYPE, IS_ACTIVE
#   - LS_FLAG, LS_DETAIL, PARTITION_OR_INDEX, PARTITION_METHOD
#   - CUSTOM_SCRIPT_PARAMS, RETENTION_DETAILS, DEPLOYMENT_SOURCE_DFG
#   - INSERTED_BY, UPDATED_BY, INSERTED_TS, UPDATED_TS
# ════════════════════════════════════════════════════════════════

In [0]:
# PARAMETERS
dbutils.widgets.text("GROUP_ID", "")
dbutils.widgets.text("TARGET_LOAD_TABLE", "")
dbutils.widgets.text("ENVIRONMENT", "dev")
dbutils.widgets.text("LOB", "")  # Line of Business filter
dbutils.widgets.text("RUN_LAYER", "")  # L0, L1, L2, or ALL

GROUP_ID = dbutils.widgets.get("GROUP_ID").strip().upper()
TARGET_TABLE = dbutils.widgets.get("TARGET_LOAD_TABLE").strip()
ENV = dbutils.widgets.get("ENVIRONMENT").strip()
LOB_FILTER = dbutils.widgets.get("LOB").strip().upper()
RUN_LAYER = dbutils.widgets.get("RUN_LAYER").strip().upper()

# Determine layer from GROUP_ID suffix or RUN_LAYER parameter
if RUN_LAYER:
    LAYER = RUN_LAYER
elif GROUP_ID.endswith("_L0"):
    LAYER = "L0"
elif GROUP_ID.endswith("_L1"):
    LAYER = "L1"
elif GROUP_ID.endswith("_L2"):
    LAYER = "L2"
else:
    LAYER = "ALL"

if not GROUP_ID:
    raise ValueError("GROUP_ID is required")

print(f"GROUP_ID      : {GROUP_ID}")
print(f"LAYER         : {LAYER}")
print(f"TARGET_TABLE  : {TARGET_TABLE or 'ALL'}")
print(f"LOB_FILTER    : {LOB_FILTER or 'ALL'}")
print(f"ENVIRONMENT   : {ENV}")

In [0]:
# IMPORTS
import traceback
import json
import re
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configuration
CATALOG = "demo_catalog"
CONTROL_SCHEMA = "admin"

print(f"CATALOG       : {CATALOG}")
print(f"CONTROL_SCHEMA: {CONTROL_SCHEMA}")

In [0]:
# READ SOURCE - Supports all formats
def read_source(url, fmt="csv", delimiter=",", custom_params=None):
    """
    Read from HTTP/S3/DBFS/Volumes/Delta into Spark DataFrame.
    Supports: csv, json, parquet, delta, excel, avro, orc
    
    Args:
        url: Source URL or path
        fmt: File format
        delimiter: CSV delimiter
        custom_params: JSON string with additional read options
    """
    fmt = (fmt or "csv").strip().lower()
    url = url.strip()
    
    if not url:
        raise ValueError("Source URL is empty")
    
    print(f"  Reading [{fmt}] from: {url[:80]}...")
    
    # Parse custom parameters
    read_options = {}
    if custom_params:
        try:
            read_options = json.loads(custom_params)
            print(f"  Custom read options: {read_options}")
        except:
            print(f"  ⚠ Failed to parse custom_params, ignoring")
    
    # HTTP/HTTPS sources
    if url.startswith("http"):
        import requests, io, pandas as pd
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        
        if fmt == "csv":
            pdf = pd.read_csv(io.BytesIO(response.content), sep=delimiter, **read_options)
            return spark.createDataFrame(pdf)
        elif fmt == "json":
            pdf = pd.read_json(io.BytesIO(response.content), **read_options)
            return spark.createDataFrame(pdf)
        elif fmt == "parquet":
            pdf = pd.read_parquet(io.BytesIO(response.content), **read_options)
            return spark.createDataFrame(pdf)
        elif fmt in ["xlsx", "xls", "excel"]:
            pdf = pd.read_excel(io.BytesIO(response.content), **read_options)
            return spark.createDataFrame(pdf)
        else:
            raise ValueError(f"Unsupported HTTP format: {fmt}")
    
    # Spark native reads for Delta, Parquet, etc.
    reader = spark.read.format(fmt)
    
    if fmt == "csv":
        reader = reader.option("header", "true").option("inferSchema", "true").option("delimiter", delimiter)
    elif fmt == "json":
        reader = reader.option("multiLine", "true")
    
    # Apply custom read options
    for key, val in read_options.items():
        reader = reader.option(key, val)
    
    return reader.load(url)

In [0]:
# WRITE TABLE - Supports partitioning, retention, and all load types
def write_table(df, catalog, schema, table, load_type="FULL", merge_keys=None, 
                partition_cols=None, retention_days=None):
    """
    Write DataFrame to Delta table with advanced options.
    
    Args:
        df: Source DataFrame
        catalog, schema, table: Target location
        load_type: FULL, INCREMENTAL, APPEND, MERGE, SCD2
        merge_keys: Comma-separated merge key columns
        partition_cols: Comma-separated partition columns
        retention_days: Data retention in days (deletes older records)
    
    Returns:
        Row count written
    """
    # Create schema if needed
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    
    full_name = f"{catalog}.{schema}.{table}"
    load_type = (load_type or "FULL").strip().upper()
    
    # Parse partition columns
    part_cols = [c.strip() for c in (partition_cols or "").split(",") if c.strip()]
    
    # FULL LOAD - Overwrite
    if load_type == "FULL":
        writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if part_cols:
            writer = writer.partitionBy(*part_cols)
        writer.saveAsTable(full_name)
        count = df.count()
    
    # INCREMENTAL/APPEND - Append new records
    elif load_type in ("INCREMENTAL", "APPEND"):
        writer = df.write.format("delta").mode("append").option("mergeSchema", "true")
        if part_cols:
            writer = writer.partitionBy(*part_cols)
        writer.saveAsTable(full_name)
        count = df.count()
    
    # MERGE - Upsert based on keys
    elif load_type == "MERGE":
        if not merge_keys:
            raise ValueError(f"MERGE_KEYS required for MERGE load type")
        
        keys = [k.strip() for k in merge_keys.split(",")]
        temp_view = f"_tmp_{table}_{datetime.now().strftime('%Y%m%d%H%M%S')}"
        df.createOrReplaceTempView(temp_view)
        
        # Create table if not exists
        create_writer = df.limit(0).write.format("delta").mode("append")
        if part_cols:
            create_writer = create_writer.partitionBy(*part_cols)
        create_writer.saveAsTable(full_name)
        
        # Build merge condition
        merge_cond = " AND ".join([f"target.{k} = source.{k}" for k in keys])
        
        # Get all columns except keys for UPDATE
        update_cols = [c for c in df.columns if c not in keys]
        update_set = ", ".join([f"target.{c} = source.{c}" for c in update_cols])
        
        # Execute MERGE
        merge_sql = f"""
            MERGE INTO {full_name} AS target
            USING {temp_view} AS source
            ON {merge_cond}
            WHEN MATCHED THEN UPDATE SET {update_set}
            WHEN NOT MATCHED THEN INSERT *
        """
        spark.sql(merge_sql)
        count = df.count()
    
    # SCD2 - Slowly Changing Dimension Type 2
    elif load_type == "SCD2":
        if not merge_keys:
            raise ValueError(f"MERGE_KEYS required for SCD2 load type")
        
        keys = [k.strip() for k in merge_keys.split(",")]
        temp_view = f"_tmp_{table}_{datetime.now().strftime('%Y%m%d%H%M%S')}"
        
        # Add SCD2 columns if not present
        if "scd_start_date" not in df.columns:
            df = df.withColumn("scd_start_date", F.current_timestamp())
        if "scd_end_date" not in df.columns:
            df = df.withColumn("scd_end_date", F.lit(None).cast(TimestampType()))
        if "is_current" not in df.columns:
            df = df.withColumn("is_current", F.lit(True))
        
        df.createOrReplaceTempView(temp_view)
        
        # Create table if not exists
        create_writer = df.limit(0).write.format("delta").mode("append")
        if part_cols:
            create_writer = create_writer.partitionBy(*part_cols)
        create_writer.saveAsTable(full_name)
        
        # Build merge condition
        merge_cond = " AND ".join([f"target.{k} = source.{k}" for k in keys])
        
        # SCD2 MERGE: Close old records, insert new versions
        merge_sql = f"""
            MERGE INTO {full_name} AS target
            USING {temp_view} AS source
            ON {merge_cond} AND target.is_current = true
            WHEN MATCHED THEN 
                UPDATE SET 
                    target.scd_end_date = current_timestamp(),
                    target.is_current = false
            WHEN NOT MATCHED THEN INSERT *
        """
        spark.sql(merge_sql)
        count = df.count()
    
    else:
        raise ValueError(f"Unsupported LOAD_TYPE: {load_type}")
    
    # Apply retention policy if specified
    if retention_days and retention_days > 0:
        apply_retention(full_name, retention_days)
    
    return count

In [0]:
# APPLY RETENTION POLICY
def apply_retention(full_table_name, retention_days):
    """
    Delete records older than retention_days from table.
    Looks for common timestamp columns: load_ts, inserted_ts, created_ts, _etl_load_ts
    """
    try:
        # Check which timestamp column exists
        df_sample = spark.sql(f"SELECT * FROM {full_table_name} LIMIT 1")
        ts_cols = ["load_ts", "_etl_load_ts", "inserted_ts", "created_ts", "timestamp"]
        
        ts_col = None
        for col in ts_cols:
            if col in df_sample.columns:
                ts_col = col
                break
        
        if not ts_col:
            print(f"  ⚠ No timestamp column found for retention policy, skipping")
            return
        
        cutoff_date = datetime.now() - timedelta(days=retention_days)
        cutoff_str = cutoff_date.strftime("%Y-%m-%d")
        
        # Delete old records
        delete_sql = f"""
            DELETE FROM {full_table_name}
            WHERE {ts_col} < '{cutoff_str}'
        """
        spark.sql(delete_sql)
        print(f"  🗑 Retention applied: Deleted records older than {retention_days} days")
        
    except Exception as e:
        print(f"  ⚠ Retention policy failed: {str(e)}")

In [0]:
# EXECUTE GENERIC SCRIPTS
def execute_generic_script(script_code, custom_params=None):
    """
    Execute a generic Python/SQL script with parameters.
    
    Args:
        script_code: Python or SQL code to execute
        custom_params: JSON string with parameters to inject
    
    Returns:
        Result message
    """
    if not script_code or script_code.strip() == "":
        return "No script to execute"
    
    print(f"  Executing generic script...")
    
    # Parse custom parameters
    params = {}
    if custom_params:
        try:
            params = json.loads(custom_params)
            print(f"  Script parameters: {params}")
        except:
            print(f"  ⚠ Failed to parse custom_params")
    
    # Inject parameters into script (simple string replacement)
    script = script_code
    for key, val in params.items():
        placeholder = f"${{{key}}}"
        script = script.replace(placeholder, str(val))
    
    # Determine if SQL or Python
    script_upper = script.strip().upper()
    if any(script_upper.startswith(kw) for kw in ["SELECT", "INSERT", "UPDATE", "DELETE", "CREATE", "ALTER", "DROP", "MERGE"]):
        # SQL script
        result = spark.sql(script)
        if script_upper.startswith("SELECT"):
            count = result.count()
            return f"Query returned {count} rows"
        else:
            return "SQL executed successfully"
    else:
        # Python script
        exec(script, {"spark": spark, "dbutils": dbutils, "F": F, "params": params})
        return "Python script executed successfully"

In [0]:
# AUDIT LOG - Track all executions
def write_audit(group_id, table_name, layer, status, message, rows, start_time, end_time, lob=None):
    """
    Write audit record. Never raises.
    """
    try:
        safe_msg = str(message).replace("'", "''")[:400]
        start_ts = start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_ts = end_time.strftime("%Y-%m-%d %H:%M:%S")
        duration = (end_time - start_time).total_seconds()
        lob_val = lob or "UNKNOWN"
        
        spark.sql(f"""
            INSERT INTO {CATALOG}.{CONTROL_SCHEMA}.audit_log (
                DATA_FLOW_GROUP_ID,
                TARGET_TABLE,
                STATUS,
                MESSAGE,
                ETL_LAYER,
                LOB,
                ROWS_PROCESSED,
                DURATION_SECONDS,
                START_TIME,
                END_TIME,
                LOAD_TS
            )
            VALUES (
                '{group_id}',
                '{table_name}',
                '{status}',
                '{safe_msg}',
                '{layer}',
                '{lob_val}',
                {rows},
                {duration},
                '{start_ts}',
                '{end_ts}',
                current_timestamp()
            )
        """)
    except Exception as e:
        print(f"  ⚠ Audit write failed: {str(e)}")

In [0]:
# PROCESS LAYER - SUPPORTS ALL METADATA COLUMNS
def process_layer(detail_table, layer):
    """
    Process all objects for a given layer with comprehensive column support.
    
    Supported columns:
      - DATA_FLOW_GROUP_ID, LOB, SOURCE, TARGET_OBJ_SCHEMA, TARGET_OBJ_NAME
      - PRIORITY, TARGET_OBJ_TYPE, TRANSFORM_QUERY, GENERIC_SCRIPTS
      - SOURCE_PK, TARGET_PK, LOAD_TYPE, IS_ACTIVE
      - LS_FLAG, LS_DETAIL, PARTITION_OR_INDEX, PARTITION_METHOD
      - CUSTOM_SCRIPT_PARAMS, RETENTION_DETAILS, DEPLOYMENT_SOURCE_DFG
      - INSERTED_BY, UPDATED_BY, INSERTED_TS, UPDATED_TS
    """
    print(f"\n{'='*70}")
    print(f"  LAYER: {layer} | TABLE: {detail_table}")
    print(f"{'='*70}")
    
    # Validate layer
    if layer not in ["L0", "L1", "L2"]:
        raise ValueError(f"Invalid layer '{layer}'")
    
    # Build query filters
    if layer == "L0":
        obj_col = "SOURCE_OBJ_NAME"
    else:
        obj_col = "TARGET_OBJ_NAME"
    
    filters = [f"DATA_FLOW_GROUP_ID = '{GROUP_ID}'"]
    filters.append("IS_ACTIVE = 'Y'")
    
    if TARGET_TABLE and TARGET_TABLE.upper() != "ALL":
        filters.append(f"{obj_col} = '{TARGET_TABLE}'")
    
    if LOB_FILTER and LOB_FILTER.upper() != "ALL":
        filters.append(f"LOB = '{LOB_FILTER}'")
    
    where_clause = " AND ".join(filters)
    
    # Query control table
    query = f"""
        SELECT 
            DATA_FLOW_GROUP_ID,
            LOB,
            SOURCE,
            SOURCE_OBJ_SCHEMA,
            SOURCE_OBJ_NAME,
            TARGET_OBJ_SCHEMA,
            TARGET_OBJ_NAME,
            PRIORITY,
            TARGET_OBJ_TYPE,
            TRANSFORM_QUERY,
            GENERIC_SCRIPTS,
            SOURCE_PK,
            TARGET_PK,
            LOAD_TYPE,
            IS_ACTIVE,
            LS_FLAG,
            LS_DETAIL,
            PARTITION_OR_INDEX,
            PARTITION_METHOD,
            CUSTOM_SCRIPT_PARAMS,
            RETENTION_DETAILS,
            DEPLOYMENT_SOURCE_DFG,
            INSERTED_BY,
            UPDATED_BY,
            INSERTED_TS,
            UPDATED_TS
        FROM {CATALOG}.{CONTROL_SCHEMA}.{detail_table}
        WHERE {where_clause}
        ORDER BY COALESCE(PRIORITY, 999), {obj_col}
    """
    
    print(f"  Executing query...")
    
    try:
        rows = spark.sql(query).collect()
    except Exception as e:
        error_msg = f"Failed to query {detail_table}: {str(e)}"
        print(f"  ❌ {error_msg}")
        raise RuntimeError(error_msg) from e
    
    if not rows:
        print(f"  ⚠ No active objects found")
        return True
    
    print(f"  Objects to process: {len(rows)}\n")
    
    # Process each object
    failed_objects = []
    success_count = 0
    
    for idx, row in enumerate(rows, 1):
        r = row.asDict()
        t0 = datetime.now()
        status = "FAILED"
        msg = ""
        count = 0
        target_table = None
        
        try:
            # Extract common fields
            lob = (r.get("LOB") or "").strip()
            load_type = (r.get("LOAD_TYPE") or "FULL").strip().upper()
            target_obj_type = (r.get("TARGET_OBJ_TYPE") or "TABLE").strip().upper()
            generic_scripts = (r.get("GENERIC_SCRIPTS") or "").strip()
            custom_params = (r.get("CUSTOM_SCRIPT_PARAMS") or "").strip()
            ls_flag = (r.get("LS_FLAG") or "N").strip().upper()
            ls_detail = (r.get("LS_DETAIL") or "").strip()
            partition_cols = (r.get("PARTITION_OR_INDEX") or "").strip()
            partition_method = (r.get("PARTITION_METHOD") or "").strip()
            retention_str = (r.get("RETENTION_DETAILS") or "").strip()
            deployment_dfg = (r.get("DEPLOYMENT_SOURCE_DFG") or "").strip()
            
            # Parse retention days
            retention_days = None
            if retention_str:
                try:
                    retention_days = int(retention_str)
                except:
                    print(f"  ⚠ Invalid retention_details: {retention_str}")
            
            if layer == "L0":
                # ═══════════════════════════════════════════════════
                # L0: FILE INGESTION
                # ═══════════════════════════════════════════════════
                source_url = (r.get("SOURCE") or "").strip()
                target_schema = (r.get("SOURCE_OBJ_SCHEMA") or "").strip()
                target_table = (r.get("SOURCE_OBJ_NAME") or "").strip()
                file_format = (r.get("INPUT_FILE_FORMAT") or "csv").strip().lower()
                delimiter = (r.get("DELIMETER") or ",").strip()
                
                if not source_url or not target_schema or not target_table:
                    raise ValueError("SOURCE, SOURCE_OBJ_SCHEMA, SOURCE_OBJ_NAME required")
                
                # Strip file extension
                import os
                target_table = os.path.splitext(target_table)[0]
                
                full_name = f"{CATALOG}.{target_schema}.{target_table}"
                
                print(f"  [{idx}/{len(rows)}] ▶ {full_name}")
                print(f"    LOB: {lob} | Type: {target_obj_type} | Load: {load_type}")
                print(f"    Source: {source_url[:60]}...")
                
                # Execute generic script BEFORE if LS_FLAG = 'B' (Before)
                if ls_flag == "B" and generic_scripts:
                    print(f"  Executing PRE-script...")
                    execute_generic_script(generic_scripts, custom_params)
                
                # Read source
                df = read_source(source_url, file_format, delimiter, custom_params)
                
                if df is None:
                    raise ValueError("read_source returned None")
                
                # Add audit columns
                df = (
                    df
                    .withColumn("_etl_group_id", F.lit(GROUP_ID))
                    .withColumn("_etl_layer", F.lit(layer))
                    .withColumn("_etl_lob", F.lit(lob))
                    .withColumn("_etl_env", F.lit(ENV))
                    .withColumn("_etl_load_ts", F.current_timestamp())
                )
                
                # Write with partitioning and retention
                count = write_table(
                    df, CATALOG, target_schema, target_table, 
                    load_type=load_type,
                    partition_cols=partition_cols,
                    retention_days=retention_days
                )
                
                # Execute generic script AFTER if LS_FLAG = 'A' (After)
                if ls_flag == "A" and generic_scripts:
                    print(f"  Executing POST-script...")
                    execute_generic_script(generic_scripts, custom_params)
                
                status = "SUCCESS"
                msg = f"{count:,} rows loaded"
                print(f"    ✅ {msg}")
                success_count += 1
                
            else:
                # ═══════════════════════════════════════════════════
                # L1/L2: TRANSFORMATION
                # ═══════════════════════════════════════════════════
                target_schema = (r.get("TARGET_OBJ_SCHEMA") or "").strip()
                target_table = (r.get("TARGET_OBJ_NAME") or "").strip()
                transform_query = (r.get("TRANSFORM_QUERY") or "").strip()
                merge_keys = (r.get("TARGET_PK") or r.get("SOURCE_PK") or "").strip()
                source_schema = (r.get("SOURCE_OBJ_SCHEMA") or target_schema).strip()
                source_table = (r.get("SOURCE_OBJ_NAME") or "").strip()
                
                if not target_schema or not target_table:
                    raise ValueError("TARGET_OBJ_SCHEMA, TARGET_OBJ_NAME required")
                
                if not transform_query and not source_table:
                    raise ValueError("Either TRANSFORM_QUERY or SOURCE_OBJ_NAME required")
                
                full_name = f"{CATALOG}.{target_schema}.{target_table}"
                
                print(f"  [{idx}/{len(rows)}] ▶ {full_name}")
                print(f"    LOB: {lob} | Type: {target_obj_type} | Load: {load_type}")
                
                # Execute generic script BEFORE
                if ls_flag == "B" and generic_scripts:
                    print(f"  Executing PRE-script...")
                    execute_generic_script(generic_scripts, custom_params)
                
                # Execute transformation
                if transform_query:
                    print(f"  Transform: Custom SQL")
                    # Add catalog prefix if missing
                    if source_schema and f"{source_schema}." in transform_query and f"{CATALOG}.{source_schema}." not in transform_query:
                        transform_query = transform_query.replace(f"{source_schema}.", f"{CATALOG}.{source_schema}.")
                    
                    df = spark.sql(transform_query)
                else:
                    print(f"  Transform: Direct copy from {CATALOG}.{source_schema}.{source_table}")
                    df = spark.table(f"{CATALOG}.{source_schema}.{source_table}")
                
                if df is None:
                    raise ValueError("Transformation returned None")
                
                # Add audit columns
                df = (
                    df
                    .withColumn("_etl_group_id", F.lit(GROUP_ID))
                    .withColumn("_etl_layer", F.lit(layer))
                    .withColumn("_etl_lob", F.lit(lob))
                    .withColumn("_etl_env", F.lit(ENV))
                    .withColumn("_etl_load_ts", F.current_timestamp())
                )
                
                # Write with all advanced options
                count = write_table(
                    df, CATALOG, target_schema, target_table,
                    load_type=load_type,
                    merge_keys=merge_keys,
                    partition_cols=partition_cols,
                    retention_days=retention_days
                )
                
                # Execute generic script AFTER
                if ls_flag == "A" and generic_scripts:
                    print(f"  Executing POST-script...")
                    execute_generic_script(generic_scripts, custom_params)
                
                status = "SUCCESS"
                msg = f"{count:,} rows processed"
                print(f"    ✅ {msg}")
                success_count += 1
        
        except Exception as e:
            status = "FAILED"
            msg = f"{type(e).__name__}: {str(e)[:200]}"
            print(f"    ❌ {msg}")
            failed_objects.append({"table": target_table or "UNKNOWN", "error": msg})
            traceback.print_exc()
        
        finally:
            # Write audit
            t1 = datetime.now()
            write_audit(GROUP_ID, target_table or "UNKNOWN", layer, status, msg, count, t0, t1, lob)
    
    # Summary
    print(f"\n{'─'*70}")
    print(f"  Summary: {success_count}/{len(rows)} succeeded")
    
    if failed_objects:
        print(f"  ❌ Failed objects:")
        for fo in failed_objects:
            print(f"    - {fo['table']}: {fo['error'][:80]}")
        raise RuntimeError(f"{len(failed_objects)} object(s) failed. See audit log.")
    
    print(f"{'─'*70}\n")
    return True

In [0]:
# MAIN EXECUTION
start_time = datetime.now()

print(f"\n{'╔'+'═'*68+'╗'}")
print(f"║  ETL FRAMEWORK ENHANCED v3.0 - START{' '*28}║")
print(f"║  GROUP: {GROUP_ID:<57}║")
print(f"║  LAYER: {LAYER:<57}║")
print(f"║  TARGET: {(TARGET_TABLE or 'ALL'):<56}║")
print(f"║  LOB: {(LOB_FILTER or 'ALL'):<60}║")
print(f"{'╚'+'═'*68+'╝'}\n")

try:
    # Determine which control table to use
    control_table_map = {
        "L0": "data_flow_l0_detail",
        "L1": "data_flow_pb_detail",
        "L2": "data_flow_pb_detail"
    }
    
    if LAYER == "ALL":
        # Process all layers sequentially
        process_layer("data_flow_l0_detail", "L0")
        process_layer("data_flow_pb_detail", "L1")
        process_layer("data_flow_pb_detail", "L2")
    elif LAYER in control_table_map:
        # Process single layer
        process_layer(control_table_map[LAYER], LAYER)
    else:
        raise ValueError(f"Invalid LAYER: {LAYER}. Must be L0, L1, L2, or ALL.")
    
    # SUCCESS
    end_time = datetime.now()
    total_duration = (end_time - start_time).total_seconds()
    
    print(f"\n{'╔'+'═'*68+'╗'}")
    print(f"║  ✅ ETL FRAMEWORK ENHANCED v3.0 - SUCCESS{' '*22}║")
    print(f"║  Duration: {total_duration:.1f}s{' '*(55-len(str(int(total_duration))))}║")
    print(f"{'╚'+'═'*68+'╝'}\n")

except Exception as e:
    # FAIL LOUDLY
    end_time = datetime.now()
    total_duration = (end_time - start_time).total_seconds()
    error_type = type(e).__name__
    error_msg = str(e)
    
    print(f"\n{'╔'+'═'*68+'╗'}")
    print(f"║  ❌ ETL FRAMEWORK ENHANCED v3.0 - FAILED{' '*23}║")
    print(f"║  Error: {error_type:<58}║")
    print(f"║  Duration: {total_duration:.1f}s{' '*(55-len(str(int(total_duration))))}║")
    print(f"{'╚'+'═'*68+'╝'}\n")
    
    print(f"\n{'='*70}")
    print(f"ERROR DETAILS:")
    print(f"  Type: {error_type}")
    print(f"  Message: {error_msg}")
    print(f"\nQuery audit log:")
    print(f"  SELECT * FROM {CATALOG}.{CONTROL_SCHEMA}.audit_log")
    print(f"  WHERE DATA_FLOW_GROUP_ID='{GROUP_ID}'")
    print(f"  ORDER BY LOAD_TS DESC")
    print(f"\nFull traceback:")
    print(f"{'='*70}")
    traceback.print_exc()
    
    # Re-raise to fail the job
    raise RuntimeError(f"ETL Framework failed: {error_type}: {error_msg}") from e